In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
import numpy as np

from sklearn.ensemble import RandomForestRegressor
import xgboost
from pyprojroot import here

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import sys
sys.path.append(str(here("Scripts")))
from model_utils import ModelSelector, train_test_split, one_hot_encode


In [2]:
data_path = here("Data/FPL_data.parquet")
data = pd.read_parquet(here(data_path))
data.head()

,name,position,team,element,opponent_team,was_home,gw,season_id,match_date,opp,goals_scored_ema_4,assists_ema_4,bonus_ema_4,clean_sheets_ema_4,creativity_ema_4,goals_conceded_ema_4,ict_index_ema_4,influence_ema_4,own_goals_ema_4,penalties_missed_ema_4,penalties_saved_ema_4,red_cards_ema_4,threat_ema_4,yellow_cards_ema_4,expected_assists_ema_4,expected_goals_ema_4,expected_goals_conceded_ema_4,team_h_score_ema_4,team_a_score_ema_4,gf_ema_4,ga_ema_4,sh_ema_4,sht_ema_4,corners_ema_4,yc_ema_4,rc_ema_4,fouls_ema_4,goals_scored_lagged_1,assists_lagged_1,bonus_lagged_1,bps_lagged_1,clean_sheets_lagged_1,creativity_lagged_1,goals_conceded_lagged_1,ict_index_lagged_1,influence_lagged_1,own_goals_lagged_1,penalties_missed_lagged_1,penalties_saved_lagged_1,red_cards_lagged_1,saves_lagged_1,threat_lagged_1,yellow_cards_lagged_1,starts_lagged_1,expected_assists_lagged_1,expected_goal_involvements_lagged_1,expected_goals_lagged_1,expected_goals_conceded_lagged_1,team_h_score_lagged_1,team_a_score_lagged_1,gf_lagged_1,ga_lagged_1,sh_lagged_1,sht_lagged_1,corners_lagged_1,yc_lagged_1,rc_lagged_1,fouls_lagged_1,xp_lagged_1,selected_lagged_1,transfers_balance_lagged_1,transfers_in_lagged_1,transfers_out_lagged_1,value_lagged_1,prob_draw,prob_loss,prob_u25,prob_ah_line,prob_ah_opp,total_points
0,Aaron Anselmino,DEF,Chelsea,774,5,False,25,2024-25,2025-02-14,Brighton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0.277778,0.37037,0.420168,0.0,0.578035,0
1,Aaron Anselmino,DEF,Chelsea,774,2,False,26,2024-25,2025-02-22,Aston Villa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.000,0.000,0.000,3.000,8.000,0.000,9.000,2.000,0.0,15.000,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,3,0,0,3,8,0,9,2,0,15,1.8,0,0,0,0,40,0.294118,0.380228,0.420168,0.0,0.505051,0
2,Aaron Anselmino,DEF,Chelsea,774,17,True,27,2024-25,2025-02-25,Southampton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.600,0.400,0.400,2.600,10.800,2.800,6.600,2.400,0.0,15.400,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,2,1,1,2,15,7,3,3,0,16,0.5,156,82,92,10,40,0.125,0.076923,0.266667,0.0,0.518135,0
3,Aaron Anselmino,DEF,Chelsea,774,11,True,28,2024-25,2025-03-09,Leicester,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.160,0.240,1.840,1.560,14.080,5.680,5.560,1.840,0.0,11.640,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,4,0,4,0,19,10,4,1,0,6,1.0,236,29,58,29,40,0.125,0.083333,0.3125,0.0,0.512821,0
4,Aaron Anselmino,DEF,Chelsea,774,1,False,29,2024-25,2025-03-16,Arsenal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.296,0.144,1.504,0.936,16.448,6.208,8.136,1.504,0.0,11.784,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1,0,1,0,20,7,12,1,0,12,0.5,325,21,52,31,40,0.277778,0.555556,0.520833,1.0,0.485437,0


In [3]:
data.drop(columns=['element', 'gw', 'opponent_team', "ict_index_ema_4", "ict_index_lagged_1"], inplace=True)

In [4]:
X_train, y_train, X_valid, y_valid, X_test, y_test = train_test_split(data)
X_train.shape, y_train.shape, X_valid.shape, y_valid.shape, X_test.shape, y_test.shape

((81677, 74), (81677,), (27605, 74), (27605,), (18183, 74), (18183,))

In [5]:
X_train, X_valid, X_test = one_hot_encode(X_train, X_valid, X_test, cols_to_encode=["position", "was_home"])

In [6]:
X_train.head()

,goals_scored_ema_4,assists_ema_4,bonus_ema_4,clean_sheets_ema_4,creativity_ema_4,goals_conceded_ema_4,influence_ema_4,own_goals_ema_4,penalties_missed_ema_4,penalties_saved_ema_4,red_cards_ema_4,threat_ema_4,yellow_cards_ema_4,expected_assists_ema_4,expected_goals_ema_4,expected_goals_conceded_ema_4,team_h_score_ema_4,team_a_score_ema_4,gf_ema_4,ga_ema_4,sh_ema_4,sht_ema_4,corners_ema_4,yc_ema_4,rc_ema_4,fouls_ema_4,goals_scored_lagged_1,assists_lagged_1,bonus_lagged_1,bps_lagged_1,clean_sheets_lagged_1,creativity_lagged_1,goals_conceded_lagged_1,influence_lagged_1,own_goals_lagged_1,penalties_missed_lagged_1,penalties_saved_lagged_1,red_cards_lagged_1,saves_lagged_1,threat_lagged_1,yellow_cards_lagged_1,starts_lagged_1,expected_assists_lagged_1,expected_goal_involvements_lagged_1,expected_goals_lagged_1,expected_goals_conceded_lagged_1,team_h_score_lagged_1,team_a_score_lagged_1,gf_lagged_1,ga_lagged_1,sh_lagged_1,sht_lagged_1,corners_lagged_1,yc_lagged_1,rc_lagged_1,fouls_lagged_1,xp_lagged_1,selected_lagged_1,transfers_balance_lagged_1,transfers_in_lagged_1,transfers_out_lagged_1,value_lagged_1,prob_draw,prob_loss,prob_u25,prob_ah_line,prob_ah_opp,position_DEF,position_FWD,position_GK,position_MID,was_home_0.0,was_home_1.0
0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.0,0.000,0,0,0,0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0.322581,0.322581,0.653595,0.0,0.555556,0,1,0,0,1,0
1,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,1.000,2.000,2.00,1.000,14.000,8.00,6.000,1.000,0.0,7.000,0,0,0,0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1,2,2,1,14,8,6,1,0,7,2.1,15789,0,0,0,55,0.285714,0.173913,0.60241,0.0,0.512821,0,1,0,0,0,1
2,0.0,0.0,0.0,0.0,0.4400,0.0,0.0,0.0,0.0,0.0,0.0,2.400,0.0,0.0,0.0,0.0,1.400,1.200,2.00,0.600,13.600,6.00,6.400,2.200,0.0,8.600,0,0,0,-2,0,1.1,0,0.0,0,0,0,0,0,6,0,0,0.0,0.0,0.0,0.0,2,0,2,0,13,3,7,4,0,11,1.0,15599,-1682,1899,3581,55,0.3125,0.30303,0.60241,0.0,0.52356,0,1,0,0,0,1
3,0.0,0.0,0.0,0.0,0.2640,0.0,0.0,0.0,0.0,0.0,0.0,1.440,0.0,0.0,0.0,0.0,0.840,1.520,1.20,1.160,13.760,4.80,5.440,1.720,0.0,7.960,0,0,0,0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,2,0,2,14,3,4,1,0,7,0.3,15457,-737,2897,3634,55,0.3125,0.357143,0.653595,0.0,0.492611,0,1,0,0,1,0
4,0.0,0.0,0.0,0.0,0.1584,0.0,0.0,0.0,0.0,0.0,0.0,0.864,0.0,0.0,0.0,0.0,0.504,1.312,1.12,0.696,9.856,3.68,5.664,1.832,0.0,9.576,0,0,0,0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,1,1,0,4,2,6,2,0,12,0.6,12065,-3682,1857,5539,54,0.3125,0.363636,0.581395,0.0,0.505051,0,1,0,0,0,1


In [7]:
xgb = xgboost.XGBRegressor(random_state=77, scoring='neg_mean_squared_error', objective='reg:squarederror', eval_metric='rmse')
rf = RandomForestRegressor(random_state=77)

In [8]:
param_grid = [
    {   "n_estimators": np.linspace(100, 300, 3, dtype=int),
        "max_depth": np.linspace(3, 7, 3, dtype=int),
        "min_samples_split": np.linspace(2, 10, 3, dtype=int)
    },
    {   "n_estimators": np.linspace(100, 300, 3, dtype=int),
        "max_depth": np.linspace(3, 7, 3, dtype=int),
        "learning_rate": np.linspace(0.01, 0.2, 3),
        "reg_alpha": np.linspace(0, 10, 1),
        "reg_lambda": np.linspace(0, 10, 1)
        }]

In [9]:
model = [rf, xgb]
model_names = ["rf", "xgb"]


In [10]:
selector = ModelSelector(random_state=77)

In [11]:
results, best_params = selector.params_search(models=model,
                       models_names=model_names,
                       params_grid=param_grid,
                       X_train=X_train,
                       y_train=y_train,
                       cv=5,
                       n_iter=30,
                       scoring="neg_mean_squared_error")


Fitting 5 folds for each of 27 candidates, totalling 135 fits


C:\Users\russa\PycharmProjects\FPL_Points_Predicting\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 27 is smaller than n_iter=30. Running 27 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


KeyboardInterrupt: 